# TTS Exp1 — Config C: FastVieNeuTTS-0.3B LMDeploy BF16

**Yêu cầu trước khi chạy:** `llm_responses.json` đã có trên Drive (chạy `tts_exp1_generate_responses.ipynb` trước).

| | |
|---|---|
| Model | FastVieNeuTTS-0.3B |
| Backend | LMDeploy TurbomindEngine BF16 |
| Chunk size | 40 chars (production default) |
| VRAM ước tính | ~3 GB (TTS) — không cần vLLM |

**Lưu ý:** lmdeploy conflict với vllm — chạy trong session riêng (không install vllm).

**Output:** `C_LMDeploy_BF16_q001.wav` ... `C_LMDeploy_BF16_q100.wav` → lưu Drive.

## 1. Install

In [ ]:
%%capture
!pip install 'vieneu' lmdeploy soundfile

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

## 2. Mount Drive & load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os
from pathlib import Path

DRIVE_DIR  = "/content/drive/MyDrive/tts_experiment"
OUTPUT_DIR = Path(f"{DRIVE_DIR}/wav_configC")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(f"{DRIVE_DIR}/tts_eval_queries.txt", encoding="utf-8") as f:
    TEST_QUERIES = [l.strip() for l in f if l.strip()]

with open(f"{DRIVE_DIR}/llm_responses.json", encoding="utf-8") as f:
    llm_responses = json.load(f)

print(f"Queries: {len(TEST_QUERIES)}  Responses: {len(llm_responses)}")

## 3. Helpers

In [ ]:
import asyncio, re, time, threading, statistics
import numpy as np
import soundfile as sf

SAMPLE_RATE = 24_000
PUNCTUATION = re.compile(r"[.!?;:,]")

async def chunk_llm_stream(text_stream, min_size=40):
    buffer = ""
    async for token in text_stream:
        buffer += re.sub(r"\n+", " ", token)
        while len(buffer) >= min_size:
            match = None
            for m in PUNCTUATION.finditer(buffer):
                if m.start() >= min_size // 2:
                    match = m
            if match:
                cut = match.end()
                yield buffer[:cut].rstrip()
                buffer = buffer[cut:]
            elif len(buffer) > min_size * 2:
                sp = buffer.rfind(" ", min_size // 2, min_size * 2)
                if sp > 0:
                    yield buffer[:sp].rstrip()
                    buffer = buffer[sp:]
                else:
                    break
            else:
                break
    if buffer.strip():
        yield buffer.rstrip()

async def replay_as_stream(text: str, token_delay_ms: float = 20):
    for word in text.split():
        yield word + " "
        await asyncio.sleep(token_delay_ms / 1000)

def pcm_float_to_int16_bytes(audio: np.ndarray) -> bytes:
    return (audio * 32767).clip(-32768, 32767).astype(np.int16).tobytes()

def save_wav(pcm_chunks: list, path: Path):
    raw = b"".join(pcm_chunks)
    audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    sf.write(str(path), audio, SAMPLE_RATE)

async def run_benchmark(tts_instance, config_name, queries, responses, token_delay_ms=20):
    results = []
    voice = tts_instance.get_preset_voice()
    loop = asyncio.get_running_loop()

    for q_idx, query in enumerate(queries):
        if query not in responses:
            print(f"  Q{q_idx+1}: no cached response, skip")
            continue

        full_text = responses[query]
        pcm_chunks_all = []
        ttfa_ms = None
        start = time.perf_counter()

        async def text_chunks_gen():
            async for tc in chunk_llm_stream(replay_as_stream(full_text, token_delay_ms)):
                yield tc

        async for text_chunk in text_chunks_gen():
            q = asyncio.Queue()
            def _worker(tc=text_chunk):
                try:
                    for frame in tts_instance.infer_stream(tc, voice=voice):
                        loop.call_soon_threadsafe(q.put_nowait, frame)
                except Exception as e:
                    loop.call_soon_threadsafe(q.put_nowait, e)
                finally:
                    loop.call_soon_threadsafe(q.put_nowait, None)
            threading.Thread(target=_worker, daemon=True).start()
            while True:
                frame = await q.get()
                if frame is None: break
                if isinstance(frame, Exception):
                    print(f"  TTS error: {frame}"); break
                if ttfa_ms is None:
                    ttfa_ms = (time.perf_counter() - start) * 1000
                pcm_chunks_all.append(pcm_float_to_int16_bytes(frame))

        total_ms = (time.perf_counter() - start) * 1000
        wav_path = OUTPUT_DIR / f"{config_name}_q{q_idx+1:03d}.wav"
        if pcm_chunks_all:
            save_wav(pcm_chunks_all, wav_path)

        results.append({
            "config": config_name, "query_idx": q_idx + 1,
            "query": query, "response_chars": len(full_text),
            "ttfa_ms": round(ttfa_ms, 1) if ttfa_ms else None,
            "total_ms": round(total_ms, 1),
            "wav_path": str(wav_path),
        })
        if (q_idx + 1) % 10 == 0:
            print(f"  {q_idx+1}/{len(queries)} done")

    ttfas  = sorted([r["ttfa_ms"] for r in results if r["ttfa_ms"]])
    totals = sorted([r["total_ms"] for r in results])
    p95 = lambda lst: lst[min(int(len(lst)*0.95), len(lst)-1)]
    print(f"\n{config_name}: TTFA median={statistics.median(ttfas):.0f}ms p95={p95(ttfas):.0f}ms  "
          f"Total median={statistics.median(totals):.0f}ms p95={p95(totals):.0f}ms")
    return results

print("Helpers ready.")

## 4. Load Config C & benchmark

In [ ]:
from vieneu.core import FastVieNeuTTS

print("Loading Config C: FastVieNeuTTS-0.3B LMDeploy BF16...")
tts_c = FastVieNeuTTS(
    backbone_repo="pnnbao-ump/VieNeu-TTS-0.3B",
    backbone_device="cuda",
    codec_repo="neuphonic/distill-neucodec",
    codec_device="cuda",
    memory_util=0.15,
    max_batch_size=4,
    enable_triton=True,
)
# FastVieNeuTTS warmup trong __init__
print("Config C ready.")

In [ ]:
results_c = await run_benchmark(tts_c, "C_LMDeploy_BF16", TEST_QUERIES, llm_responses)

## 5. Save results

In [ ]:
results_path = f"{DRIVE_DIR}/results_configC.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results_c, f, ensure_ascii=False, indent=2)
print(f"Saved: {results_path}")
print(f"WAV files: {OUTPUT_DIR}/ ({len(list(OUTPUT_DIR.glob('*.wav')))} files)")

## 6. Listen (optional)

In [ ]:
from IPython.display import Audio, display
for i in [1, 2, 3]:
    p = OUTPUT_DIR / f"C_LMDeploy_BF16_q{i:03d}.wav"
    if p.exists():
        audio, sr = sf.read(str(p))
        print(f"Q{i}: {TEST_QUERIES[i-1]}")
        display(Audio(audio, rate=sr))